# QMIND · Visibility Thresholds

## 1. Choose one stage per session

Start with **PREFLIGHT** on a CPU runtime. Fill in the Drive and source settings below, then use **Runtime → Run all**. The notebook checks the selected dataset and shows the destination and assignment. Only the selected stage can run.

| Stage | Before starting | What you get |
|---|---|---|
| PREFLIGHT | Source bundle and Drive mount | Read-only experiment checks |
| GENERATE | Fresh GPU runtime, new run or resumable team part | CLIPDraw PNG, SVG and experiment metadata |
| MANIFEST | All team parts complete, or explicit existing image roots | Frozen list of verified images |
| JUDGE_SIGLIP2 / JUDGE_FLORENCE2 | Manifest, fresh GPU runtime for the selected judge | Checkpoints and ranked words or captions |
| TEXT_SCORE | Manifest, fresh CPU runtime, verified SigLIP commit | OCR and typography diagnostics |
| ANALYZE | Completed judge shards and text-score export | Per-image combined CSV, summaries and reports |
| ALL | Same setup as preflight | Guided session sequence; each compute stage is selected separately |

**First smoke run:** use a new run ID and `smoke_test`. Each teammate draws one assigned word with the existing smoke settings. For a research run choose a new ID and `full`. Full mode uses 256 paths, 1,000 iterations and four augmentations. Downloads and installs for a model or generator require its stage name in both fields.

In [ ]:
STAGE = "PREFLIGHT"  # @param ["PREFLIGHT","GENERATE","MANIFEST","JUDGE_SIGLIP2","JUDGE_FLORENCE2","TEXT_SCORE","ANALYZE","ALL"]
CONFIRM = ""  # @param {type:"string"}
READY = None

## 2. Mount Google Drive

Sign in with the account that owns or can access the project. Set the project folder relative to the mount. For another account, use a Drive shortcut at the **same path** to the same files. Put the verified source ZIP in this folder before starting. The notebook checks that the mount and folder exist.

In [ ]:
from pathlib import Path
import os
from google.colab import drive

MOUNT = Path("/content/drive")
PROJECT_FOLDER = "MyDrive/QMIND/Visibility Thresholds"  # @param {type:"string"}
drive.mount(str(MOUNT))
DRIVE_ROOT = (MOUNT / PROJECT_FOLDER).resolve()
if not os.path.ismount(MOUNT) or not DRIVE_ROOT.is_relative_to(MOUNT) or not DRIVE_ROOT.is_dir():
    raise ValueError("Mount Drive and select an existing project folder beneath /content/drive.")

## 3. Verify the source

From the repository root at the reviewed release commit, use a clean tracked working tree. In PowerShell, check that the first command prints the release SHA, then package the source to a new local ZIP:

```powershell
git rev-parse HEAD
python -B -m judges.prepare_colab --unified-base (git rev-parse HEAD) --output visibility-source.zip
```

Enter the reviewed **full Git commit** in `BASE_COMMIT` and the packager's **archive SHA-256** in `ARCHIVE_SHA256`. Upload that ZIP to the project folder when ready for Colab. Keep this exact archive across sessions and open the notebook from the same release.

In [ ]:
import hashlib
import io
import re
import sys
import zipfile
from pathlib import PurePosixPath

BASE_COMMIT = ""  # @param {type:"string"}
ARCHIVE_SHA256 = ""  # @param {type:"string"}
ARCHIVE = DRIVE_ROOT / "visibility-source.zip"

def verify_bundle(archive, expected_sha, base, local_root):
    if not re.fullmatch("[0-9a-f]{40}", base):
        raise ValueError("Paste the full reviewed release commit in BASE_COMMIT.")
    if not re.fullmatch("[0-9a-f]{64}", expected_sha):
        raise ValueError("Paste the archive SHA-256 printed by the local packager.")
    content = archive.read_bytes()
    if hashlib.sha256(content).hexdigest() != expected_sha:
        raise ValueError("Source ZIP differs from the pinned digest.")
    dest = local_root / expected_sha
    with zipfile.ZipFile(io.BytesIO(content)) as bundle:
        names = bundle.namelist()
        if bundle.read("base_commit.txt").decode() != base:
            raise ValueError("Source base commit differs from the reviewed commit.")
        if len(set(names)) != len(names):
            raise ValueError("Duplicate source archive entry.")
        for name in names:
            path = PurePosixPath(name)
            if path.is_absolute() or ".." in path.parts or "\\" in name or ":" in name:
                raise ValueError(f"Invalid archive member: {name}")
            target = dest / name
            if target.resolve() != target:
                raise ValueError(f"Unexpected source link: {target}")
            data = bundle.read(name)
            if target.exists():
                if target.read_bytes() != data:
                    raise ValueError(f"Local source differs at {target}; start a fresh runtime.")
            else:
                target.parent.mkdir(parents=True, exist_ok=True)
                with target.open("xb") as f:
                    f.write(data)
        extra = [p for p in dest.rglob("*") if p.is_file() and
                 "__pycache__" not in p.parts and p.relative_to(dest).as_posix() not in names]
        if extra:
            raise ValueError(f"Unexpected local source files: {extra[:3]}. Start a fresh runtime.")
    return dest

SRC = verify_bundle(ARCHIVE, ARCHIVE_SHA256, BASE_COMMIT, Path("/content/qmind_source"))
if "unified_colab" in sys.modules and Path(sys.modules["unified_colab"].__file__).parent != SRC:
    raise RuntimeError("A different source bundle was loaded. Disconnect and delete this runtime.")
sys.path.insert(0, str(SRC))
import unified_colab as vt
print("Verified working copy", ARCHIVE_SHA256, "based on", BASE_COMMIT)

## 4. Configure the experiment

**Required:** choose a persistent run ID and cohort ID. Select the generation CSV, judge vocabulary and imageability metadata. The defaults use the official 150 nouns. For a larger split, set generation to e.g. `noun_words_split/3.csv` and both other CSVs to `noun_words.csv`. A split file is already a subset; team assignment applies within that file.

**Optional:** set team size and person (1-based), shard count and shard number. For saved images, list mounted absolute part directories in `EXISTING_ROOTS` and start at MANIFEST. Select full or smoke folders deliberately. Keep IDs, dataset choices, team size, shard count and source fixed across sessions. Only the active person or shard changes. Use one writer per part or shard; initialize shared environment locks once before parallel team runs.

In [ ]:
RUN_ID = "smoke01"  # @param {type:"string"}
COHORT_ID = "pilot"  # @param {type:"string"}
GENERATION_MODE = "smoke_test"  # @param ["smoke_test","full"]
GENERATION_CSV = "sampled_words.csv"
VOCABULARY_CSV = "sampled_words.csv"
METADATA_CSV = "sampled_words.csv"

PERSON, NUM_PEOPLE = 1, 1
SHARD, NUM_SHARDS = 1, 1
EXISTING_ROOTS = []  # Example: [str(DRIVE_ROOT / "existing-images" / "part1")]
RESUME = False
TEXT_REVISION = ""  # Full commit for google/siglip-so400m-patch14-384, needed only for TEXT_SCORE.

READY = None
selected = vt.select_stage(STAGE, CONFIRM)
if selected == "GENERATE" and EXISTING_ROOTS:
    raise ValueError("Use MANIFEST to evaluate existing images.")
if selected == "TEXT_SCORE" and not re.fullmatch("[0-9a-f]{40}", TEXT_REVISION):
    raise ValueError("Verify the text scorer's SigLIP commit and enter it in TEXT_REVISION.")
if any(not Path(p).is_absolute() or not Path(p).resolve().is_relative_to(MOUNT) for p in EXISTING_ROOTS):
    raise ValueError("Existing image roots must be absolute paths on the mounted Drive.")
if type(SHARD) is not int or type(NUM_SHARDS) is not int or not 1 <= SHARD <= NUM_SHARDS:
    raise ValueError("Expected 1 <= SHARD <= NUM_SHARDS.")
if selected in ("GENERATE", "JUDGE_SIGLIP2", "JUDGE_FLORENCE2", "TEXT_SCORE"):
    if globals().get("COMPUTE_SESSION", selected) != selected:
        raise RuntimeError("Select a fresh runtime before changing compute stages.")
    COMPUTE_SESSION = selected

processed = SRC / "dataset/processed"
flow = vt.Workflow(drive_root=DRIVE_ROOT, run_id=RUN_ID, cohort_id=COHORT_ID,
                   generation_csv=processed / GENERATION_CSV, vocabulary_csv=processed / VOCABULARY_CSV,
                   metadata_csv=processed / METADATA_CSV, source_hash=ARCHIVE_SHA256,
                   input_roots=EXISTING_ROOTS, people=NUM_PEOPLE, mode=GENERATION_MODE, shards=NUM_SHARDS)
print("Person", PERSON, "assigned", len(flow.assignment(PERSON)), "words; first five:", flow.assignment(PERSON)[:5])
print("Selected inputs:", flow.cfg.input_roots, "\nManifest:", flow.manifest_path)

## 5. Validate dataset and stage prerequisites

The validator checks the committed noun pool, sample and six splits. A checkout without the generated `cleaned_words.csv` reports **incomplete**. The notebook shows that limitation and accepts this specific missing file when the tracked inputs pass. Problems with other required data stop execution. Preflight leaves Drive outputs untouched.

In [ ]:
READY = None
vt.dataset_check(processed)
flow.bind(selected)
if selected in ("PREFLIGHT", "ALL"):
    print("Text-score revision:", TEXT_REVISION or "required before TEXT_SCORE")
    if flow.manifest_path.exists():
        print("Verified existing manifest:", len(flow.manifest()["images"]), "images")
    else:
        print("Manifest pending. Generate all intended parts or select existing roots, then run MANIFEST.")
    if selected == "ALL":
        print("Sessions: GENERATE (GPU) -> MANIFEST -> JUDGE_SIGLIP2 (GPU) -> JUDGE_FLORENCE2 (GPU)"
              " -> TEXT_SCORE (CPU) -> ANALYZE. Reopen from section 1 on each new runtime.")
READY = (selected, id(flow))

def active(stage):
    if READY != (vt.select_stage(STAGE, CONFIRM), id(flow)):
        raise RuntimeError("Rerun configuration and validation after changing the stage.")
    return selected == stage

## 6. Generate CLIPDraw images · GPU

GENERATE runs the original CLIPDraw setup and runtime checks, including compiling DiffVG, then calls the original batch runner. It can download sources and model weights. Use a fresh GPU runtime and the part assigned to you. Set `RESUME=True` only for an interrupted part with the same settings. After verification, disconnect and delete the generation runtime before judging.

In [ ]:
if active("GENERATE"):
    flow.generate(PERSON, RESUME)

## 7. Freeze or verify the image manifest

MANIFEST requires a usable PNG and completion metadata for every selected image. For notebook-generated parts it also checks all team completion receipts. Existing identical manifests can be verified again. A changed cohort requires a new run ID. Keep source images at the mounted paths shown in section 4.

In [ ]:
if active("MANIFEST"):
    snapshot = flow.manifest(publish=True)
    print(len(snapshot["images"]), "verified images at", flow.manifest_path)

## 8. Evaluate with SigLIP2 · fresh GPU

Ranks the configured vocabulary for each image. Setup downloads the isolated judge environment and pinned model. Set the shard number before launch. Checkpoints and one full export per shard are saved to Drive.

In [ ]:
if active("JUDGE_SIGLIP2"):
    flow.judge("siglip2", SHARD, RESUME)

## 9. Evaluate with Florence-2 · fresh GPU

Produces captions using the pinned Florence model and custom code, with its own Transformers environment. Use the same cohort and shard count as SigLIP2. Reopen this notebook in a fresh GPU runtime and rerun setup from section 1.

In [ ]:
if active("JUDGE_FLORENCE2"):
    flow.judge("florence2", SHARD, RESUME)

## 10. Score visible text · fresh CPU

Before selecting TEXT_SCORE, independently verify a full Hugging Face commit for **google/siglip-so400m-patch14-384** and enter it in section 4. This model differs from the recognition judge. This stage can download the CPU environment, OCR weights and SigLIP weights. It scores the whole manifest once and saves text metrics by image ID. Its Linux dependency lock and model caches persist on Drive.

In [ ]:
if active("TEXT_SCORE"):
    flow.text_score(TEXT_REVISION, RESUME)

## 11. Analyze completed judge exports · CPU

Production analysis requires the frozen manifest, completed SigLIP2 and Florence-2 shards, and the completed text-score export with its checkpoints. Run on CPU. The stage verifies inputs and writes the existing judge comparison plus `combined.csv`, `combined_summary.json` and `combined_report.md` under `analysis/`.

For an **exploratory snapshot**, set the option below to True. All manifest images appear. Completed judge shards and checksum-verified text checkpoints contribute observations. Missing sources have presence flags and explicit missing IDs. The judge-only report uses images with both judge results. Inspect failed jobs before interpreting coverage. Each mode creates a fresh directory once; preserve or archive an existing report before requesting another snapshot. Exploratory outputs go under `analysis_exploratory/`.

In [ ]:
EXPLORATORY_PARTIAL = False  # @param {type:"boolean"}
if active("ANALYZE"):
    flow.analyze(partial=EXPLORATORY_PARTIAL)

## 12. Locate results and recover a run

Reopen the same source archive and mount the same project folder. Match section 4 to the original run. **A missing completion file means the stage is pending or partial.** Inspect its checkpoint directory or batch summary. For an interrupted compute stage, select that stage, set its confirmation, set `RESUME=True` and use the same person or shard number. For damaged generation files, preserve the failed part for inspection and choose a new run ID.

Each run has a `run.json` configuration, `images/partN/` folders for generation and `generation/partN.json` receipts. Stage artifacts are under `stages/<cohort>/`. Completed exports are read by ANALYZE; starting another condition requires a new run. Disconnect the runtime when finished.

In [ ]:
print("Run:", flow.run)
print("Generation receipts:", sorted(p.name for p in (flow.run / "generation").glob("part*.json")))
for label, path in [("Manifest", flow.manifest_path),
                    ("Text scores", flow.artifacts / "text_score/results.json"),
                    ("Combined production report", flow.artifacts / "analysis/combined_report.md"),
                    ("Exploratory snapshot", flow.artifacts / "analysis_exploratory/combined_report.md")]:
    print(label, "available" if path.is_file() else "pending", path)
for model in vt.PINS:
    exports = sorted((flow.artifacts / "judges" / model / COHORT_ID).glob("shard*/full/complete.json"))
    print(model, len(exports), "completed shard exports of", NUM_SHARDS)
    for export in exports:
        print(" ", export.with_name("results.json"))